# பாடம் 16 - Microsoft Foundry உடன் அதிக அளவிலான முகவர்கள்

இந்த நோட்ட்புக் இல் நீங்கள் **Contoso** எனப்படும் கற்பனை நிறுவனத்திற்கான **முன்னணி தயாரிப்பு முகவர்** ஒன்றை கட்டமைக்கின்றீர்கள். முந்தைய பாடங்களுடன் மாறாக, இங்கு முகவர் விவேகச் சுற்றம் முக்கியம் அல்ல — முகவரைப் பாதுகாப்பாக அளவிலவு இயக்குவதற்கான அவற்றைச் சுற்றும் அனைத்தும் முக்கியம்:

1. **கருவி அழைப்பு** — ஆர்டர் தேடல்கள் மற்றும் டிக்கெட் உருவாக்கம்.
2. **RAG** — அறிவுத் தளத்திலிருந்து கொள்கை பதில்கள்.
3. **நினைவகம்** — வாடிக்கையாளரை மடக்கி நினைவில் வைத்தல்.
4. **மாதிரி வழி அமைப்பு** — எளிய கோரிக்கைகளை சிறிய மாதிரிக்கு, சிக்கலானவைகளை பெரிய மாதிரிக்கு அனுப்பு.
5. **பதிலின் காசிங்** — மாதிரி அழைப்பில்லாமல் மீண்டும் மீண்டும் கேட்கும் கேள்விகளுக்கு பதில்.
6. **மனித அனுமதி** — ஒரு அளவை மீறிய திருப்பிகள் ஒப்புதலுக்காக நிறுத்தப்படுகின்றன.
7. **மதிப்பீட்டு வாசல்** — ஒரு ஆன்லைனுக்கு வெளியான சோதனை தொகுப்பு ஒரு மோசமான வெளியீட்டை தடுத்தல்.
8. **கவனிக்கக்கூடிய தன்மை** — ஒவ்வொரு கோரிக்கையின் சுற்றிலும் OpenTelemetry தடயப்படம்.

ஒவ்வொரு பகுதியும் தனியாக உள்ளது மற்றும் இயங்கக்கூடியது. ஒவ்வொரு வரியையும் படியுங்கள் — தயாரிப்பு அடிப்படைகள் ஆராய்ச்சிக்கேற்ப சிறியவையாக வைக்கப்பட்டுள்ளன.


## அமைப்பு

இந்த நோட்புக் இயக்குவதற்கு முன், நீங்கள் கீழ்காணும் நிபந்தனைகளை பூர்த்தி செய்திருக்க வேண்டும்:

1. **ஒரு Microsoft Foundry திட்டம்** மற்றும் அதில் அமையப்பட்ட ஒரு உரையாடல் மாதிரி (எ.கா `gpt-4.1-mini`).
2. **Azure CLIயை பயன்படுத்தி பதிவு செய்திருத்தல்** — உங்கள் டெர்மினலில் `az login` ஓட்டவும்.
3. **தேவைப்படும் சூழல் மாறிலிகளை அமைத்திருத்தல்:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Microsoft Foundry திட்டத்தின் அடைவை.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் அமையப்பட்ட மாதிரியின் பெயர்.

RAG பகுதி `AZURE_SEARCH_SERVICE_ENDPOINT` மற்றும் `AZURE_SEARCH_API_KEY` அமைக்கப்பட்டுள்ளபோது **Azure AI Search** ஐ பயன்படுத்துகிறது, மற்றும் இல்லாவிட்டால் நோட்புக் தேடல் வளம் இல்லாமலேயே இயங்க நினைவக தேடலுக்கு மாற்றம் ஆகிறது.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. கருவிகள்

உற்பத்தி கருவிகள் உண்மையான அமைப்புகளுக்கெதிராக உண்மையான பணியைச் செய்கிறன. இங்கே நாம் சாதாரண Python செயல்பாடுகளைப் பயன்படுத்தி ஒரு ஆர்டர் தரவுத்தளத்தையும் ஒரு டிக்கெட் அமைப்பையும் சிமுலேட் செய்கிறோம். `@tool` அலங்கரிப்பாளர் அவற்றை முகவருக்கு வெளிப்படுத்துகிறது.

`issue_refund` ஒரு நிலைதக்கத்திற்கு மேல் பணத்தை திரும்பப்பெற `approval_mode="always_require"` என்பதை பயன்படுத்துவதை கவனியுங்கள் — இது நாம் பின்னர் அமைக்கும் மனிதன்-மூடுபனி முறைத்தன்மை ஆகும்.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — கொள்கை அறிவுக் கோப்பகம்

கொள்கை தொடர்பான கேள்விகளுக்கு ("உங்கள் திரும்பும் காலம் எவ்வளவு?") மாதிரியின் நினைவகத்திலிருந்து அல்லாமல், அதிகாரபூர்வமான மூலத்திலிருந்து பதிலளிக்க வேண்டும். ஒரு சிறிய அறிவுக் கோப்பகத்தை தேடல் கருவியாக பயன்படுத்துகின்றோம்.

தயாரிப்பில் இது **Azure AI Search** ஆகும்; இங்கே நாங்கள் நினைவக உள்ளே கூர்மையான சொல் தேடலை வழங்குகிறோம், இதனால் நோட்புக் எங்கும் இயங்கும், மற்றும் சுற்றுப்புற மாறிலிகள் உள்ள போது தானாகவே Azure AI Search-க்கு மாறுவோம்.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. நினைவகம்

யாருடன் பேசிக்கொண்டிருக்கிறதென மறந்துவிடும் ஆதரவு முகவர் ஒரு மோசமான ஆதரவு முகவரென்றே ஆகும். நாங்கள் வாடிக்கையாளர் ஒருவருக்கொரு சிறிய சுருக்கப்பட்ட சுயவிவர சேமிப்பகத்தை வைத்திருக்கிறோம் மற்றும் முகவரின் வழிமுறைகளில் ஒரு குறுகிய சுருக்கத்தை இணைக்கிறோம். உற்பத்தியில் இது ஒரு நினைவக சேவையைக் குறிக்கும் (பாடம் 13 ஐப் பார்); இங்கே ஒரு dict எனும் தொகுப்பு முறைபாட்டைப் புலப்படுத்துகிறது.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. மாதிரி வழியனுப்பல் மற்றும் பதிலளிப்பு கேஷிங்  

ஒரு தனித்த கோரிக்கையாளர் கையாள்பவருக்கு இணைக்கப்பட்ட இரண்டு செலவு கட்டிகள்:  

- **வழியனுப்பல்**: ஒரு எளிய சூறாவளி வகைப்பவை ஒரு கோரிக்கைக்கு சிறிய மாதிரி தேவைப்படுகிறதா அல்லது பெரிய மாதிரி தேவைப்படுகிறதா என்று தீர்மானிக்கிறது.  
- **கேஷிங்**: சாதாரணப்படுத்தப்பட்ட மீண்டும் வரும் கேள்விகள் மாதிரி அழைப்பு இல்லாமல் நேரடியாக கேஷிலிருந்து வழங்கப்படுகின்றன.  

இங்கு வகைப்பவை செயற்கையாக எளிமையாக உள்ளது. உற்பத்தியில் நீங்கள் இதை போக்குவரத்துக்கு எதிராக சோதித்து Foundry-இன் மாதிரி வழியனுப்பியுடன் மாற்றலாம்.  


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. முகவர், மனித ஒப்புதல் மற்றும் கண்காணிப்பு

இப்போது மேல் கூறிய கருவிகளிலிருந்து முகவரை தொகுத்து, ஒவ்வொரு கோரிக்கையையும் OpenTelemetry ஸ்பானில் சுற்றி தொடங்குகிறோம். `handle_support_request` செயல்பாடு உற்பத்தி கோரிக்கை கையாள்பவர்: cache → route → trace → run → cache.

மனித ஒப்புதல் கட்டமைப்பால் கையாளப்படுகிறது: `issue_refund` `approval_mode="always_require"` என்பதால், இயக்கம் இடைநிறுத்தப்பட்டு, நாம் தெளிவாக தீர்மானிக்கும் ஒப்புதல் கோரிக்கையை வெளிப்படுத்துகிறது.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. மதிப்பீடு வாயில்

இது பாடத்தின் வெளியீட்டு வாயில்: ஒரு ஆஃப்லைன் சோதனை தொகுப்பு ஏஜண்டை மதிப்பீடு செய்கிறது, மற்றும் போக்குவரத்து வெற்றிகரமாக முந்தைய எல்லைக்குள் வந்தால் மட்டுமே Deployment தொடர்கிறது. இங்கு மதிப்பெண் வழங்குபவர் ஒரு எளிய முக்கிய சொல்-ஒட்டி சோதனை, இதன் மூலம் நோட்புக் சுயபூரணமாக இருக்கும்; உற்பத்தியில் நீங்கள் LLM-ஐ நீதிபதியாக அல்லது ஒரு கட்டமைப்பு மதிப்பீட்டாளராக பயன்படுத்துவீர்கள் (பாடம் 10 ஐப் பாருங்கள்).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ஒன்றாகக் கூடியது: ஒரு ஒப்படைக்கப்பட்ட வெளியீடு

கீழே உள்ள செலில் பாடம் விவரிக்கின்ற முழு சுழற்சியை காட்டுகிறது: மதிப்பீடு வாயிலை இயக்குங்கள், அது கடக்கப்பட்டால் மட்டுமே "வெளியிடவும்". இது Foundry Agent சேவைக்கு ஒரு முகவரின் பதிப்பை முன்னிலைப்படுத்துவதற்கு முன் CI இல் நீங்கள் இயங்கவேண்டிய நடைமுறை ஆகும்.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## சுருக்கம்

நீங்கள் ஒவ்வொரு செயல்பாட்டு கவலைகளும் இணைக்கப்பட்டு தயாரான வாடிக்கையாளர் ஆதரவு முகவரியை உருவாக்கினீர்கள்:

- **கருவிகள், RAG, மற்றும் நினைவகம்** முகவரிக்கு திறனும் சூழலும் வழங்குகின்றன.
- **மாதிரி வழிமாற்றல் மற்றும் சேமிப்பு** தாமதமும் செலவும் கட்டுப்பாட்டில் வைத்திருக்கிறன.
- **மனித ஒப்புதல்** பெரிய பண திரும்பியல்கள் போன்ற உயர் ஆபத்தான நடவடிக்கைகள் பாதுகாக்கபடுகின்றன.
- **மதிப்பீட்டு வாயில்** மோசமான வெளியீடுகளை கப்பலில் இருந்து தடுக்கும்.
- **கண்காணிப்பு** ஒவ்வொரு கோரிக்கையையும் படிக்கக்கூடியதாக வைக்கும்.

### சவால்

இந்த முகவரியை விரிவாக்கவும்:

1. **பல மாதிரிகளுக்கு ஆதரவு** — மூன்றாவது "கருத்து" நிலையைச் சேர்த்து, மேம்படுத்தல்கள்/பயன்பாடுகளை அதற்கு வழிமாற்றவும்.
2. **மதிப்பீட்டு வாயில்களைச் சேர்க்கவும்** — `TEST_CASES` ஐ பணம் திரும்பியலை ஒப்புதலுக்கு உபயோகப்படுத்தும் நிலைகளுடன் விரிவாக்கி, வாயில் பிழைகளைக் கண்டறிதலை உறுதி செய்யவும்.
3. **செலவை கவனித்த வழிமாற்றல்** — ஒவ்வொரு கோரிக்கைக்கும் (சிறிய, பெரிய, சேமிப்பு) மதிப்பீடு செய்த செலவை பதிவு செய்து, கலவையான கேள்விகளுக்குப்பிறகு செலவு அறிக்கையை அச்சிடவும்.

அடுத்த പാഠத்தில், நீங்கள் முற்றிலும் உங்கள் சொந்த இயந்திரத்தில் Microsoft Foundry Local மற்றும் Qwen கொண்டு முகவரியை இயக்குவீர்கள்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
